# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
import joblib


## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [3]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X = X.copy()
        X['hour'] = pd.to_datetime(X['timestamp']).dt.hour
        X['dayofweek'] = pd.to_datetime(X['timestamp']).dt.dayofweek
        X = X.drop(columns=['timestamp'])
        return X


In [4]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_col):
        self.target_col = target_col
        self.ohe = OneHotEncoder(sparse_output=False, dtype=int)
        self.cat_cols = None

    def fit(self, X, y=None):
        X = X.copy()
        self.cat_cols = list(X.select_dtypes(include=['object', 'str']).columns)
        if self.target_col in self.cat_cols:
            self.cat_cols.remove(self.target_col)
        self.ohe.fit(X[self.cat_cols])
        return self

    def transform(self, X, y=None):
        X = X.copy()
        if self.target_col in X.columns:
            y = X[self.target_col]
            X = X.drop(columns=[self.target_col])
        encoded = self.ohe.transform(X[self.cat_cols])
        feature_names = self.ohe.get_feature_names_out(self.cat_cols)
        df_encoded = pd.DataFrame(encoded, columns=feature_names, index=X.index)
        X = pd.concat([X.drop(columns=self.cat_cols), df_encoded], axis=1)
        if y is not None:
            X[self.target_col] = y
        return X


In [5]:
class TrainValidationTest:
    def __init__(self, test_size=0.2, random_state=21):
        self.test_size = test_size
        self.random_state = random_state

    def split(self, X, y):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.test_size, random_state=self.random_state, stratify=y
        )
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train, y_train, test_size=self.test_size, random_state=self.random_state, stratify=y_train
        )
        return X_train, X_valid, X_test, y_train, y_valid, y_test


## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [6]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.best_models = []

    def choose(self, X_train, y_train, X_valid, y_valid):
        self.best_models = []
        best_overall = None
        best_overall_score = 0
        best_overall_name = ''

        for idx, gs in enumerate(self.grids):
            model_name = self.grid_dict[idx]
            print(f'Estimator: {model_name}')
            gs.fit(X_train, y_train)
            best_gs = gs.best_estimator_
            valid_score = accuracy_score(y_valid, best_gs.predict(X_valid))

            self.best_models.append({
                'model': model_name,
                'params': str(gs.best_params_),
                'valid_score': valid_score
            })

            print(f'Best params: {gs.best_params_}')
            print(f'Best training accuracy: {gs.best_score_:.3f}')
            print(f'Validation set accuracy score for best params: {valid_score:.3f}')
            print()

            if valid_score > best_overall_score:
                best_overall_score = valid_score
                best_overall = best_gs
                best_overall_name = model_name

        print(f'Classifier with best validation set accuracy: {best_overall_name}')
        return best_overall

    def best_results(self):
        return pd.DataFrame(self.best_models)


## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [7]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f'Accuracy of the final model is {acc}')
        return acc

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f'Model successfully saved to {path}')


## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [8]:
# 1. Load data
df = pd.read_csv('../../src/data/checker_submits.csv')

# 2. Create preprocessing pipeline
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('dayofweek'))
])

# 3. Fit and transform
data = preprocessing.fit_transform(df)
y = data['dayofweek']
X = data.drop(columns=['dayofweek'])

# 4. Split
tvt = TrainValidationTest()
X_train, X_valid, X_test, y_train, y_valid, y_test = tvt.split(X, y)

# 5. Model selection
jobs = -1

svm_params = [{
    'kernel': ('linear', 'rbf', 'sigmoid'),
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ('balanced', None),
    'random_state': [21],
    'probability': [True]
}]
svm = SVC()
gs_svm = GridSearchCV(svm, svm_params, scoring='accuracy', cv=2, n_jobs=jobs)

dt_params = [{
    'max_depth': range(1, 50),
    'class_weight': ('balanced', None),
    'criterion': ('gini', 'entropy'),
    'random_state': [21]
}]
dt = DecisionTreeClassifier()
gs_dt = GridSearchCV(dt, dt_params, scoring='accuracy', cv=2, n_jobs=jobs)

rf_params = [{
    'n_estimators': [5, 10, 50, 100],
    'max_depth': range(1, 50),
    'class_weight': ('balanced', None),
    'criterion': ('gini', 'entropy'),
    'random_state': [21]
}]
rf = RandomForestClassifier()
gs_rf = GridSearchCV(rf, rf_params, scoring='accuracy', cv=2, n_jobs=jobs)

grids = [gs_svm, gs_dt, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

ms = ModelSelection(grids, grid_dict)
best_model = ms.choose(X_train, y_train, X_valid, y_valid)

print()
print('Best results:')
print(ms.best_results())

# 6. Finalize
final = Finalize(best_model)
final.final_score(X_train, y_train, X_test, y_test)
final.save_model(f'{best_model.__class__.__name__}_{accuracy_score(y_test, best_model.predict(X_test))}.sav')


Estimator: SVM


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878

Estimator: Decision Tree
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.863

Estimator: Random Forest
Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.904

Classifier with best validation set accuracy: Random Forest

Best results:
           model                                             params  \
0            SVM  {'C': 10, 'class_weight': None, 'gamma': 'auto...   
1  Decision Tree  {'class_weight': 'balanced', 'criterion': 'gin...   
2  Random Forest  {'class_weight': None, 'criterion': 'gini', 'm...   

   v

In [9]:
# Verify saved model
loaded_model = joblib.load("RandomForestClassifier_0.9112426035502958.sav")
final_loaded = Finalize(loaded_model)
score = final_loaded.final_score(X_train, y_train, X_test, y_test)
print(f"Loaded model gives the same score: {score}")


Accuracy of the final model is 0.9112426035502958
Loaded model gives the same score: 0.9112426035502958
